# 22. End-to-End Case Study: Used Car Market Fair-Value Regression

A complete regression pipeline demonstrating exponential depreciation modeling, log transforms, and target encoding.


## 1. Objective
Build an end-to-end regression model to estimate fair market vehicle value (`selling_price`).
Solve key challenges:
1. Exponential price depreciation with vehicle age and mileage.
2. High-cardinality vehicle models (48 levels).
3. Right-skewed target distribution.


## 2. Dataset & Decision Context
- **Dataset**: `used_cars.csv` (22,000 listings across 8 brands, 48 models, 10 locations)
- **ML Objective**: Minimize Root Mean Squared Error (RMSE) on test listings.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

df = pd.read_csv('../datasets/used_cars/used_cars.csv')
print(f"Used Cars Dataset: {df.shape[0]:,} rows")


## 3. Data Cleaning & Feature Engineering


In [ ]:
# 1. Clean domain bound errors
df_clean = df[(df['mileage'] > 0) & (df['engine_cc'] > 0)].copy()

# 2. Impute Categorical Missingness as 'Unknown'
df_clean['service_history'] = df_clean['service_history'].fillna('Unknown')
df_clean['accident_history'] = df_clean['accident_history'].fillna('Unknown')

# 3. Engineer Domain Features
df_clean['car_age'] = (2024 - df_clean['year']).clip(lower=0.5)
df_clean['mileage_per_year'] = df_clean['mileage'] / df_clean['car_age']
df_clean['reg_delay'] = df_clean['registration_year'] - df_clean['year']

# 4. Target Transformation
X = df_clean.drop(columns=['car_id', 'selling_price'])
y = np.log(df_clean['selling_price'])

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


## 4. Building the Leak-Proof Preprocessing & Modeling Pipeline


In [ ]:
cat_high = ['model', 'location', 'brand']
cat_low = ['fuel_type', 'transmission', 'service_history', 'accident_history']
num_cols = ['year', 'car_age', 'mileage', 'mileage_per_year', 'engine_cc', 'owner_count', 'reg_delay']

preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', TargetEncoder(cv=KFold(n_splits=5, shuffle=True, random_state=42), smooth="auto"), cat_high),
        ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_low),
        ('num', 'passthrough', num_cols)
    ]
)

pipe = Pipeline([
    ('prep', preprocessor),
    ('regressor', HistGradientBoostingRegressor(random_state=42, max_iter=200))
])

pipe.fit(X_train, y_train_log)

# Evaluate in Original Dollars
preds_log = pipe.predict(X_test)
preds_dollars = np.exp(preds_log)
actual_dollars = np.exp(y_test_log)

rmse = np.sqrt(mean_squared_error(actual_dollars, preds_dollars))
r2 = r2_score(actual_dollars, preds_dollars)
mape = mean_absolute_percentage_error(actual_dollars, preds_dollars)

print("=" * 60)
print("FAIR-VALUE REGRESSION PERFORMANCE ON TEST SET:")
print(f" - Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f" - Mean Absolute % Error (MAPE):   {mape:.2%}")
print(f" - R² Score:                       {r2:.4f}")
print("=" * 60)


## 5. Predicted vs Actual Price Diagnostic


In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(actual_dollars, preds_dollars, alpha=0.25, color='#2b5c8f')
plt.plot([0, 200000], [0, 200000], color='red', linestyle='--', label='Perfect Calibration Line')
plt.title(f'Actual vs Predicted Used Car Price (R² = {r2:.4f})')
plt.xlabel('Actual Selling Price ($)')
plt.ylabel('Predicted Selling Price ($)')
plt.xlim(0, 200000)
plt.ylim(0, 200000)
plt.legend()
plt.tight_layout()
plt.show()
